In [ ]:
#from torch import nn
import pandas as pd
import numpy as np
from pathlib import Path
import torch
from nfl.lib import enums

from nfl.data_management.DataManager import DataManager
from nfl.NeuralNetwork.EPA_Predictor import EPAPredictor
from nfl.NeuralNetwork.NNSolver import Solver
from nfl.lib.utils import create_train_test_split
from torch.utils.tensorboard import SummaryWriter

In [ ]:
SCENARIO_COLS = [
    "yardline_100",
    "game_seconds_remaining",
    "has_turf",
    "temp",
    "wind",
    "has_roof",
    "ydstogo",
    "goal_to_go",
    "score_differential",
    "down",
    "div_game",
    "day_of_season",
    "series"
]

PLAY_COLS = [
    "play_type",
    "pass_location",
#    "pass_length",
#    "run_location",
    "run_gap",
    "shotgun",
    "no_huddle",
    "qb_kneel",
    "qb_spike",
    "qb_scramble",
    "air_yards"
]

RESULT_COLS = [
    "epa",
    "wpa",
    "success",
    "result",
    "series_success",
    "tackle_for_loss",
    "saftey",
    "yards_gained",
    "touchdown",
    "fumble",
    "complete_pass",
    "rushing_yards",
    "fumble_lost",
    "interception",
    "sack",
    "penalty_yards",
]

EXCLUDED_PLAY_TYPES = {
    enums.PlayType.KICK,
    enums.PlayType.EXTRA_POINT,
    enums.PlayType.NO_PLAY,
    enums.PlayType.GAME_START,
}

In [ ]:
data = DataManager.get_data(path_to_json = (Path.cwd() / "nfl/data").resolve())
data = DataManager.clean_data(data, excluded_play_types=EXCLUDED_PLAY_TYPES, target_col="epa")
data, feature_cols = DataManager.prepare_features(data, scenario_columns=SCENARIO_COLS, play_columns=PLAY_COLS)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cuda device count: ", torch.cuda.device_count())
print(f"Using {device} device")

In [ ]:
pd.set_option('display.max_columns', None)
data.head(5)

In [ ]:
split = create_train_test_split(df=data, feature_cols=feature_cols, target_col="epa", train_frac = 0.8, device=device)
X_train, y_train, X_test, y_test, scaler = split

In [ ]:
EPA_INPUT_SIZE = len(feature_cols)
NUM_ITER = 5
NUM_EPOCH = 50
EPS = 1e-8
BETAS = (0.9, 0.999)

MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS = 1, 20
MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE = 16, 256
MIN_LEARNING_RATE, MAX_LEARNING_RATE = 1e-5, 1e-2
MIN_BATCH_SIZE, MAX_BATCH_SIZE = 64, 512
MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY = 1e-5, 1e-2

BATCH_SIZE = np.linspace(MIN_BATCH_SIZE, MAX_BATCH_SIZE, num=NUM_ITER).astype(int).tolist()
LEARNING_RATE = np.geomspace(MIN_LEARNING_RATE, MAX_LEARNING_RATE, num=NUM_ITER).tolist()
NUM_HIDDEN_LAYERS = np.linspace(MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS, num=NUM_ITER).astype(int).tolist()
HIDDEN_SIZE = np.linspace(MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE, num=NUM_ITER).astype(int).tolist()
WEIGHT_DECAY = np.geomspace(MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY, num=NUM_ITER).tolist()

In [ ]:
from torch.utils.tensorboard import SummaryWriter

sweep_results = []

for bs in BATCH_SIZE:
    for lr in LEARNING_RATE:
        for hl in NUM_HIDDEN_LAYERS:
            for hs in HIDDEN_SIZE:
                for wd in WEIGHT_DECAY:
                    run_name = f"bs{bs}_lr{lr:.1e}_hl{hl}_hs{hs}_wd{wd:.1e}"
                    run_dir = f"runs/epa_hyperparameter_sweep/{run_name}"

                    # Dedicated writer per run
                    run_writer = SummaryWriter(log_dir=run_dir)

                    hparams = {
                        "batch_size": int(bs),
                        "lr": float(lr),
                        "num_hidden_layers": int(hl),
                        "hidden_size": int(hs),
                        "weight_decay": float(wd),
                    }

                    model = EPAPredictor(
                        input_size=EPA_INPUT_SIZE,
                        num_hidden_layers=int(hl),
                        hidden_size=int(hs),
                    ).to(device)

                    optimizer = torch.optim.Adam(
                        model.parameters(),
                        lr=float(lr),
                        betas=BETAS,
                        eps=EPS,
                        weight_decay=float(wd),
                    )

                    solver = Solver(
                        model=model,
                        device=device,
                        num_epochs=NUM_EPOCH,
                        batch_size=bs,
                        optimizer=optimizer,
                        criterion=torch.nn.MSELoss().to(device),
                        writer=run_writer,
                        run_name=run_name,
                        hparams=hparams,
                    )

                    metrics = solver.train(
                        X_train,
                        y_train,
                        X_test,
                        y_test,
                        modelName=run_name,
                        saveBest=True,
                    )
                    run_writer.close()

                    # Merge hyperparams and metrics into one row
                    sweep_results.append(
                        {"run_name": run_name, **hparams, **metrics}
                    )

# Convert to DataFrame and save
results_df = pd.DataFrame(sweep_results)
results_df.to_csv(
    "runs/epa_hyperparameter_sweep/sweep_results.csv", index=False
)

In [ ]:
top_models = results_df.sort_values("best_val_loss").head(5)

# Average validation loss grouped by learning rate
lr_impact = results_df.groupby("lr")["best_val_loss"].agg(["mean", "std", "min"])

# Rank hyperparameter correlations with validation loss
correlations = (
    results_df[
        [
            "batch_size",
            "lr",
            "num_hidden_layers",
            "hidden_size",
            "weight_decay",
            "best_val_loss",
        ]
    ]
    .corr()["best_val_loss"]
    .drop("best_val_loss")
)